# **Job Seekers Dataset Preprocessing**

In [ ]:
%pip install pandas
%pip install neo4j
%pip install python-dotenv

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/technical_candidates_dataset_10000.csv")

In [ ]:
df.head()

In [ ]:
#plot pie chart for job titles
import matplotlib.pyplot as plt

# Count occurrences of each level
level_counts = df["job_title"].value_counts()
# Plot pie chart
plt.figure()
level_counts.plot.pie(autopct='%1.1f%%')

plt.title("Distribution of Jobs")
plt.ylabel("")  # removes y-axis label
plt.show()

In [ ]:
#plot pie chart for job titles
import matplotlib.pyplot as plt

# Count occurrences of each level
level_counts = df["seniority_level"].value_counts()
# Plot pie chart
plt.figure()
level_counts.plot.pie(autopct='%1.1f%%')

plt.title("Distribution of Experience Levels")
plt.ylabel("")  # removes y-axis label
plt.show()

In [ ]:
# Step 1: Initial split from the raw ',' separated string
df['Skill_List'] = df['skills'].str.split(';')
df = df.explode('Skill_List')
df['Skill_List'] = df['Skill_List'].str.strip()


In [ ]:
df.info()

In [ ]:
df.to_csv("candidates_exploded_dataset.csv")

# **KG Implementation**

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase


load_dotenv()

# --- Configuration ---
URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
driver = GraphDatabase.driver(URI, auth=AUTH)

In [ ]:
import pandas as pd



FILE_PATH = "/content/candidates_exploded_dataset.csv"
df2 = pd.read_csv(FILE_PATH)


def create_jobseeker(tx, c_id, title, experience, skill):
    query = """
    MERGE (js:JobSeeker {id: $c_id})
    SET js.job_title = $title,
        js.experience_level = $experience

    MERGE (s:Skill {name: $skill})

    MERGE (js)-[:HAS_SKILL]->(s)
    """
    tx.run(query, c_id=c_id, title=title, experience=experience, skill=skill)


# Execute query
with driver.session() as session:
    for i, row in df2.iterrows():
        print(f"Processing row {i}")
        candidate_id = int(row["candidate_id"])
        title = row["job_title"]
        experience_level = row["seniority_level"]
        skill = row["Skill_List"]

        # Skip empty skills
        if pd.isna(skill) or skill.strip() == "":
            continue

        session.execute_write(create_jobseeker, candidate_id, title, experience_level, skill.strip())

driver.close()
